# Task 2 합성: X종 + N종 균형 (v3)

N종(18종) 누끼 추출 + X종(pills/) + N종(pills_n18/) 혼합 합성

합성 규칙: bbox 겹침 방지 · 클래스 중복 금지 · cycling pool · 각도증강 · 그림자 · N종 리사이즈

## 셀 1. SAM 설치 및 체크포인트

In [1]:
!pip install -q segment-anything
import os
SAM_CKPT_PATH = "/content/sam_vit_h_4b8939.pth"
if not os.path.exists(SAM_CKPT_PATH):
    !wget -q --show-progress \
        https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth \
        -O {SAM_CKPT_PATH}
    print("SAM 다운로드 완료")
else:
    print("SAM 이미 존재")

/content/sam_vit_h_ 100%[===================>]   2.39G   248MB/s    in 9.9s    
SAM 다운로드 완료


## 셀 2. 임포트 및 Drive 마운트

In [2]:
import os, json, glob, random, zipfile, io
import numpy as np
import cv2
from PIL import Image
from collections import defaultdict, Counter
from google.colab import drive
drive.mount('/content/drive')
print("Drive 마운트 완료")
from segment_anything import sam_model_registry, SamPredictor
print("SAM 임포트 완료")

Mounted at /content/drive
Drive 마운트 완료


ModuleNotFoundError: No module named 'segment_anything'

## 셀 3. 경로 설정

In [3]:
DRIVE_BASE   = "/content/drive/MyDrive/(((((((내꺼)))))) 코드잇/스프린트 미션&피드백/초급프로젝트/dataset"  # ← 수정
PILLS_X_DIR  = os.path.join(DRIVE_BASE, "pills")
PILLS_N_DIR  = os.path.join(DRIVE_BASE, "pills_n18")
BG_DIR       = os.path.join(DRIVE_BASE, "backgrounds")
DATASET_DIR  = "/content/dataset"
ZIP2_PATH    = os.path.join(DRIVE_BASE, "train_56_45_merged_coco_20260703.zip")
CLEANED2_PATH= os.path.join(DRIVE_BASE, "train_56_45_cleaned.json")
CLEANED_N_PATH= os.path.join(DRIVE_BASE, "hidden_n18_cleaned.json")
ZIP_N_PATH   = os.path.join(DRIVE_BASE, "hidden_n18_aihub_train_import_20260703.zip")
SAM_CHECKPOINT = "/content/sam_vit_h_4b8939.pth"

CLUSTER = {
    "top_left"   : (230, 330, 30, 40), "top_center" : (465, 330, 30, 40),
    "top_right"  : (700, 330, 35, 40), "bot_left"   : (245, 890, 35, 60),
    "bot_center" : (465, 1000, 30, 40),"bot_right"  : (715, 840, 40, 50),
}
LAYOUT_3 = [["top_left","top_right","bot_center"],["top_center","bot_left","bot_right"]]
LAYOUT_4 = [["top_left","top_right","bot_left","bot_right"]]
SHADOW_OFFSET=(12,12); SHADOW_BLUR=18; SHADOW_OPACITY=0.45; BLEND_FEATHER=3
TARGET_COUNT=35; ROTATE_RANGE=15; RESIZE_JITTER=0.10
OVERLAP_MARGIN=10; MAX_POS_RETRIES=5; SCALE_FACTOR=0.85

for name, p in [("PILLS_X_DIR",PILLS_X_DIR),("BG_DIR",BG_DIR),
                ("ZIP2_PATH",ZIP2_PATH),("CLEANED2_PATH",CLEANED2_PATH)]:
    print(f"{name:20s}: {'OK' if os.path.exists(p) else 'MISSING'}")

TASK2_IMG = os.path.join(DRIVE_BASE, "task2_synthesized", "images")
TASK2_ANN = os.path.join(DRIVE_BASE, "task2_synthesized", "annotations")

PILLS_X_DIR         : OK
BG_DIR              : OK
ZIP2_PATH           : OK
CLEANED2_PATH       : OK


## 셀 4. 데이터 로드

In [13]:
# N종 cleaned annotation
with open(CLEANED_N_PATH, encoding="utf-8") as f:
    cleaned_n = json.load(f)
id2img_n  = {img["id"]: img for img in cleaned_n["images"]}
id2name_n = {cat["id"]: str(cat["id"]) for cat in cleaned_n["categories"]}

print(f"N종 이미지: {len(cleaned_n['images'])}장 / {len(cleaned_n['categories'])}종")

# 배경 이미지
bg_paths = sorted(glob.glob(os.path.join(BG_DIR, "bg_*.png")))
print(f"배경 이미지: {len(bg_paths)}장")

N종 이미지: 1750장 / 18종
배경 이미지: 230장


## 셀 5. fold0에서 X종 현재 출현 횟수 계산

In [9]:
import os, shutil

ZIP_DRIVE = f"/content/drive/MyDrive/1팀 공유 문서/ai12-level1-project/dataset_5fold.zip"
DATASET_DIR = "/content/dataset"

if not os.path.exists(DATASET_DIR) or not os.listdir(DATASET_DIR):
    os.makedirs(DATASET_DIR, exist_ok=True)
    os.system(f"unzip -qo {ZIP_DRIVE} -d {DATASET_DIR}")
    print("압축 해제 완료")
else:
    print("이미 존재")

압축 해제 완료


In [7]:
import zipfile

ZIP_DRIVE = f"/content/drive/MyDrive/1팀 공유 문서/ai12-level1-project/dataset_5fold.zip"
DATASET_DIR = "/content/dataset"

with zipfile.ZipFile(ZIP_DRIVE, 'r') as z:
    for f in z.namelist()[:20]:
        print(f)

fold0/
fold1/
fold2/
fold3/
fold4/
label_map.json
fold3/train/
fold3/valid/
fold3/valid/K-003483-016232-020877-034597_0_2_0_2_90_000_200.png
fold3/valid/K-003351-019232-035206_0_2_0_2_75_000_200.png
fold3/valid/K-001900-016548-019607-029451_0_2_0_2_70_000_200.png
fold3/valid/K-003483-016232-020877-034597_0_2_0_2_75_000_200.png
fold3/valid/K-003351-016262-031863_0_2_0_2_70_000_200.png
fold3/valid/K-003351-016232-020014_0_2_0_2_75_000_200.png
fold3/valid/K-003483-025367-027733-035206_0_2_0_2_90_000_200.png
fold3/valid/K-003351-003832-022074_0_2_0_2_75_000_200.png
fold3/valid/K-003483-020238-027733-028763_0_2_0_2_75_000_200.png
fold3/valid/K-003483-016232-020877-030308_0_2_0_2_70_000_200.png
fold3/valid/K-003351-019232-022074_0_2_0_2_70_000_200.png
fold3/valid/K-003483-016232-020877-030308_0_2_0_2_75_000_200.png


In [8]:
import os

result = os.system(f"unzip -qo '{ZIP_DRIVE}' -d {DATASET_DIR}")
print("return code:", result)  # 0이면 성공

# 확인
print(os.listdir(DATASET_DIR))

return code: 0
['fold4', 'fold0', 'label_map.json', 'fold3', 'fold2', 'fold1']


In [9]:
x_current=defaultdict(int); id2name_x={}
for split in ["train","valid"]:
    jp=os.path.join(DATASET_DIR,"fold0",split,"_annotations.coco.json")
    with open(jp,encoding="utf-8") as f: coco=json.load(f)
    id2n={cat["id"]:cat["name"] for cat in coco["categories"] if cat["id"]!=0}
    id2name_x.update(id2n)
    for ann in coco["annotations"]:
        nm=id2n.get(ann["category_id"])
        if nm: x_current[nm]+=1
x_needed={n:TARGET_COUNT-c for n,c in x_current.items() if c<TARGET_COUNT}
print(f"X종 전체: {len(x_current)}개 / 추가 필요: {len(x_needed)}개 / 총 {sum(x_needed.values())}회")

# N종 필요 횟수 (모두 0에서 시작)
n_needed = {str(cat["id"]): TARGET_COUNT for cat in cleaned_n["categories"]}
print(f"N종 필요: {len(n_needed)}개 클래스 / 총 {sum(n_needed.values())}회")

X종 전체: 56개 / 추가 필요: 53개 / 총 1323회
N종 필요: 18개 클래스 / 총 630회


## 셀 6. SAM + 합성 함수 정의

In [24]:
def load_sam(checkpoint=SAM_CHECKPOINT, model_type="vit_h", device="cuda"):
    sam = sam_model_registry[model_type](checkpoint=checkpoint)
    sam.to(device=device)
    print(f"SAM 로드 완료 | {model_type} | {device}")
    return SamPredictor(sam)

def segment_pill(predictor, image_bgr, bbox_xywh):
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    predictor.set_image(image_rgb)
    x, y, w, h = [float(v) for v in bbox_xywh]
    box = np.array([x, y, x+w, y+h])
    masks, scores, _ = predictor.predict(box=box[None,:], multimask_output=True)
    return masks[int(np.argmax(scores))]

def crop_pill_rgba(image_bgr, mask, bbox_xywh, padding=5):
    H, W = image_bgr.shape[:2]
    x, y, w, h = [int(v) for v in bbox_xywh]
    x1,y1 = max(0,x-padding), max(0,y-padding)
    x2,y2 = min(W,x+w+padding), min(H,y+h+padding)
    crop_bgr  = image_bgr[y1:y2,x1:x2].copy()
    crop_mask = mask[y1:y2,x1:x2].astype(np.uint8)*255
    k = BLEND_FEATHER*2+1
    crop_mask = cv2.GaussianBlur(crop_mask,(k,k),BLEND_FEATHER)
    rgba = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2BGRA)
    rgba[:,:,3] = crop_mask
    return rgba

def boxes_overlap(b1, b2, margin=OVERLAP_MARGIN):
    x1,y1,w1,h1=b1; x2,y2,w2,h2=b2
    return not(x1+w1+margin<x2 or x2+w2+margin<x1 or y1+h1+margin<y2 or y2+h2+margin<y1)

def any_overlap(new_bbox, placed):
    return any(boxes_overlap(new_bbox, b) for b in placed)

def create_cycling_pool(pool):
    cyc={}
    for cat,paths in pool.items():
        s=paths.copy(); random.shuffle(s)
        cyc[cat]={'paths':s,'idx':0}
    return cyc

def sample_cycling(cyc, cat):
    if isinstance(cyc, dict) and 'paths' in cyc:
        e = cyc  # 이미 entry가 넘어온 경우
    else:
        e = cyc[cat]  # 전체 pool이 넘어온 경우
    p = e['paths'][e['idx']]; e['idx'] += 1
    if e['idx'] >= len(e['paths']):
        e['idx'] = 0; random.shuffle(e['paths'])
    return p

def rotate_pill(rgba, angle):
    h,w=rgba.shape[:2]
    M=cv2.getRotationMatrix2D((w/2,h/2),angle,1.0)
    cos,sin=abs(M[0,0]),abs(M[0,1])
    nw,nh=int(h*sin+w*cos),int(h*cos+w*sin)
    M[0,2]+=(nw-w)/2; M[1,2]+=(nh-h)/2
    return cv2.warpAffine(rgba,M,(nw,nh),flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_CONSTANT,borderValue=(0,0,0,0))

def sample_position(key):
    cx,cy,sx,sy=CLUSTER[key]
    x=int(np.clip(np.random.normal(cx,sx),cx-2*sx,cx+2*sx))
    y=int(np.clip(np.random.normal(cy,sy),cy-2*sy,cy+2*sy))
    return x,y

def make_shadow(rgba, bg_shape, px, py):
    H,W=bg_shape[:2]; ph,pw=rgba.shape[:2]
    alpha=rgba[:,:,3].astype(np.float32)/255.0
    dx,dy=SHADOW_OFFSET; smap=np.zeros((H,W),np.float32)
    sx,sy=px+dx,py+dy
    sx1,sy1=max(0,sx),max(0,sy); sx2,sy2=min(W,sx+pw),min(H,sy+ph)
    ax1,ay1=sx1-sx,sy1-sy; ax2,ay2=ax1+(sx2-sx1),ay1+(sy2-sy1)
    if sx2>sx1 and sy2>sy1: smap[sy1:sy2,sx1:sx2]=alpha[ay1:ay2,ax1:ax2]
    k=SHADOW_BLUR*2+1; smap=cv2.GaussianBlur(smap,(k,k),SHADOW_BLUR/3)
    return smap*SHADOW_OPACITY

def paste_pill(bg_float, rgba, cx, cy, bbox_margin=10):
    H,W=bg_float.shape[:2]; ph,pw=rgba.shape[:2]
    px,py=cx-pw//2,cy-ph//2
    px1,py1=max(0,px),max(0,py); px2,py2=min(W,px+pw),min(H,py+ph)
    ax1,ay1=px1-px,py1-py; ax2,ay2=ax1+(px2-px1),ay1+(py2-py1)
    if px2<=px1 or py2<=py1: return bg_float,None
    crop=rgba[ay1:ay2,ax1:ax2]
    alpha=crop[:,:,3:4].astype(np.float32)/255.0
    rgb=cv2.cvtColor(crop,cv2.COLOR_BGRA2BGR).astype(np.float32)
    result=bg_float.copy()
    result[py1:py2,px1:px2]=rgb*alpha+result[py1:py2,px1:px2]*(1-alpha)
    ys, xs = np.where(rgba[:,:,3] > 10)
    if len(xs) > 0:
        bbox = [int(xs.min())+px - bbox_margin,
                int(ys.min())+py - bbox_margin,
                int(xs.max())-int(xs.min()) + bbox_margin*2,
                int(ys.max())-int(ys.min()) + bbox_margin*2]
    else:
        bbox = [px, py, pw, ph]
    return result, bbox

def get_pill_actual_size(rgba):
    ys,xs=np.where(rgba[:,:,3]>10)
    if len(xs)==0: return rgba.shape[1],rgba.shape[0]
    return int(xs.max()-xs.min()),int(ys.max()-ys.min())

def analyze_x_pill_sizes(pills_dir):
    widths,heights=[],[]
    for cls_dir in os.listdir(pills_dir):
        if not cls_dir.startswith("class_"): continue
        for fn in os.listdir(os.path.join(pills_dir,cls_dir)):
            if not fn.endswith(".png"): continue
            rgba=cv2.imread(os.path.join(pills_dir,cls_dir,fn),cv2.IMREAD_UNCHANGED)
            if rgba is None or rgba.shape[2]!=4: continue
            w,h=get_pill_actual_size(rgba); widths.append(w); heights.append(h)
    stats={'median_w':int(np.median(widths)),'median_h':int(np.median(heights)),
           'mean_w':int(np.mean(widths)),'mean_h':int(np.mean(heights))}
    print(f"X종 크기 분석 (n={len(widths)})")
    print(f"  너비: mean={stats['mean_w']}  median={stats['median_w']}")
    print(f"  높이: mean={stats['mean_h']}  median={stats['median_h']}")
    return stats

def resize_n_pill(rgba, tw, th, jitter=RESIZE_JITTER):
    cw,ch=get_pill_actual_size(rgba)
    if cw==0 or ch==0: return rgba
    scale=min(tw/cw,th/ch)*np.random.uniform(1-jitter,1+jitter)
    nw=max(1,int(rgba.shape[1]*scale)); nh=max(1,int(rgba.shape[0]*scale))
    return cv2.resize(rgba,(nw,nh),interpolation=cv2.INTER_LINEAR)

def build_pill_pool(pills_dir):
    pool={}
    for cls_dir in sorted(os.listdir(pills_dir)):
        if not cls_dir.startswith("class_"): continue
        cat=cls_dir[len("class_"):]
        paths=sorted([os.path.join(pills_dir,cls_dir,f)
                      for f in os.listdir(os.path.join(pills_dir,cls_dir))
                      if f.endswith(".png")])
        if paths: pool[cat]=paths
    print(f"pool: {len(pool)}개 클래스, 총 {sum(len(v) for v in pool.values())}개")
    return pool

def weighted_sample_class(needed):
    cats=list(needed.keys()); ws=[needed[c] for c in cats]
    return str(np.random.choice(cats, p=[w/sum(ws) for w in ws]))

def fill_slots(layout, combined_needed, all_pools, all_cyc, n_cats):
    used=set(); slots=[]
    for key in layout:
        avail={k:v for k,v in combined_needed.items() if k not in used}
        if avail:
            cat=weighted_sample_class(avail)
        else:
            unused=[k for k in all_pools if k not in used]
            if not unused: break
            cat=random.choice(unused)
        used.add(cat)
        is_n=cat in n_cats
        path=sample_cycling(all_cyc[cat],cat) if cat in all_cyc else random.choice(all_pools[cat])
        slots.append((key,path,cat,is_n))
    return slots

MAX_PILL_W = int(976 * 0.40)   # 390px
MAX_PILL_H = int(1280 * 0.40)  # 512px

def clip_pill_size(rgba, max_w=MAX_PILL_W, max_h=MAX_PILL_H):
    curr_w, curr_h = get_pill_actual_size(rgba)
    if curr_w <= max_w and curr_h <= max_h:
        return rgba
    scale = min(max_w/curr_w, max_h/curr_h)
    nw = max(1, int(rgba.shape[1]*scale))
    nh = max(1, int(rgba.shape[0]*scale))
    return cv2.resize(rgba, (nw, nh), interpolation=cv2.INTER_LINEAR)

def synthesize_one(bg_path, slots, cat2label, x_size_stats=None):
    bg=cv2.imread(bg_path); canvas=bg.astype(np.float32)
    ann_list=[]; placed=[]
    for key,pill_path,cat,is_n in slots:
        rgba=cv2.imread(pill_path,cv2.IMREAD_UNCHANGED)
        if rgba is None or rgba.ndim<3 or rgba.shape[2]!=4: continue
        if is_n and x_size_stats:
            rgba=resize_n_pill(rgba,x_size_stats['median_w'],x_size_stats['median_h'])
        rgba=clip_pill_size(rgba)  # 상한선 초과 시 축소
        rgba=rotate_pill(rgba,np.random.uniform(-ROTATE_RANGE,ROTATE_RANGE))
        done=False; cur=rgba.copy()
        for phase in range(2):
            if phase==1:
                h,w=cur.shape[:2]
                cur=cv2.resize(cur,(max(1,int(w*SCALE_FACTOR)),max(1,int(h*SCALE_FACTOR))),
                               interpolation=cv2.INTER_LINEAR)
            for _ in range(MAX_POS_RETRIES):
                cx,cy=sample_position(key); ph,pw=cur.shape[:2]
                est=[cx-pw//2,cy-ph//2,pw,ph]
                if not any_overlap(est,placed):
                    px,py=cx-pw//2,cy-ph//2
                    shadow=make_shadow(cur,canvas.shape,px,py)
                    canvas=canvas*(1-shadow[:,:,np.newaxis])
                    canvas,bbox=paste_pill(canvas,cur,cx,cy)
                    if bbox is not None:
                        placed.append(bbox)
                        ann_list.append({"category_id":cat2label[cat],
                                         "bbox":[float(v) for v in bbox],
                                         "area":float(bbox[2]*bbox[3]),"iscrowd":0})
                    done=True; break
            if done: break
    return canvas.astype(np.uint8),ann_list

def run_synthesis(combined_needed, all_pools, all_cyc, bg_paths, n_cats,
                  cat2label, x_size_stats, out_img, out_ann, max_images, seed):
    random.seed(seed); np.random.seed(seed)
    os.makedirs(out_img,exist_ok=True); os.makedirs(out_ann,exist_ok=True)
    imgs,anns=[],[]; img_id,ann_id=1,1
    for i in range(max_images):
        if not combined_needed:
            print(f"\n[완료] 모든 클래스 목표 달성 ({i}장)"); break
        layout=random.choice(LAYOUT_3*2+LAYOUT_4)
        slots=fill_slots(layout,combined_needed,all_pools,all_cyc,n_cats)
        if not slots: continue
        try:
            img,ann_list=synthesize_one(random.choice(bg_paths),slots,cat2label,x_size_stats)
        except Exception as e:
            print(f"\n[ERROR]: {e}"); continue
        name=f"syn_{img_id:05d}.png"
        cv2.imwrite(os.path.join(out_img,name),img)
        H,W=img.shape[:2]
        imgs.append({"id":img_id,"file_name":name,"width":W,"height":H})
        for ann in ann_list:
            ann["id"]=ann_id; ann["image_id"]=img_id; anns.append(ann); ann_id+=1
        for _,_,cat,_ in slots:
            if cat in combined_needed:
                combined_needed[cat]-=1
                if combined_needed[cat]<=0: del combined_needed[cat]
        img_id+=1
        if i%10==0: print(f"[{i+1}/{max_images}] 남은: {len(combined_needed)}개",end="\r")
    if combined_needed: print(f"\n[경고] 미달성 {len(combined_needed)}개")
    print(f"\n합성 완료: {img_id-1}장")
    cats=[{"id":0,"name":"pill","supercategory":"none"}]
    cats+=[{"id":v,"name":k,"supercategory":"pill"} for k,v in sorted(cat2label.items(),key=lambda x:x[1])]
    coco={"images":imgs,"annotations":anns,"categories":cats}
    p=os.path.join(out_ann,"_annotations.coco.json")
    with open(p,"w") as f: json.dump(coco,f,ensure_ascii=False)
    print(f"저장: {p}")
    return coco

print("함수 정의 완료")

함수 정의 완료


## 셀 7. N종 누끼 추출 실행

> 약 **15~30분** 소요 (1750장 기준)

In [ ]:
def extract_pills_n(predictor, cleaned_n, id2img_n,
                    zip_path, output_dir=PILLS_N_DIR, padding=5):
    class_counts = defaultdict(int)
    os.makedirs(output_dir, exist_ok=True)
    total = len(cleaned_n["annotations"])
    with zipfile.ZipFile(zip_path, "r") as z:
        for i, ann in enumerate(cleaned_n["annotations"]):
            print(f"[{i+1}/{total}]", end="\r")
            fn       = id2img_n[ann["image_id"]]["file_name"]
            cat_name = str(ann["category_id"])
            try:
                img_bytes = z.read(
                    f"working/aihub_prepared/hidden_train_import/images/{fn}")
                img_pil   = Image.open(io.BytesIO(img_bytes)).convert("RGB")
                image_bgr = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)
                mask      = segment_pill(predictor, image_bgr, ann["bbox"])
                rgba      = crop_pill_rgba(image_bgr, mask, ann["bbox"], padding)
                cls_dir   = os.path.join(output_dir, f"class_{cat_name}")
                os.makedirs(cls_dir, exist_ok=True)
                idx       = class_counts[cat_name]
                cv2.imwrite(os.path.join(cls_dir, f"pill_{idx:04d}.png"), rgba)
                class_counts[cat_name] += 1
            except Exception as e:
                print(f"\n[ERROR] {fn}: {e}")
    print(f"\nN종 누끼 추출 완료: 총 {sum(class_counts.values())}개")
    return dict(class_counts)

predictor = load_sam(SAM_CHECKPOINT)
n_class_counts = extract_pills_n(predictor, cleaned_n, id2img_n, ZIP_N_PATH)

## 셀 8. X종 크기 분석 (N종 리사이즈 기준)

In [17]:
x_size_stats = analyze_x_pill_sizes(PILLS_X_DIR)

X종 크기 분석 (n=2016)
  너비: mean=230  median=209
  높이: mean=280  median=241


## 셀 9. Combined cat2label 구성 (X 56 + N 18 = 74종)

In [11]:
with open(os.path.join(DATASET_DIR,"fold0","train","_annotations.coco.json"),
          encoding="utf-8") as f:
    fold0_coco = json.load(f)

combined_cat2label = {cat["name"]: cat["id"]
                      for cat in fold0_coco["categories"] if cat["id"] != 0}
max_id = max(combined_cat2label.values())
nxt = max_id + 1
for cat in cleaned_n["categories"]:
    nm = str(cat["id"])
    if nm not in combined_cat2label:
        combined_cat2label[nm] = nxt; nxt += 1

print(f"X종: {max_id}개 → N종 추가 후: {len(combined_cat2label)}개")

X종: 56개 → N종 추가 후: 74개


## 셀 10. 합성 실행

> X종 부족분 + N종 0→35 동시 달성

In [26]:
x_pool    = build_pill_pool(PILLS_X_DIR)
n_pool    = build_pill_pool(PILLS_N_DIR)
x_cyc     = create_cycling_pool(x_pool)
n_cyc     = create_cycling_pool(n_pool)
n_cats    = set(n_pool.keys())
all_pools = {**x_pool, **n_pool}
all_cyc   = {**x_cyc, **n_cyc}

combined_needed = {}
combined_needed.update({k:v for k,v in x_needed.items() if v>0 and k in x_pool})
combined_needed.update({k:v for k,v in n_needed.items() if v>0 and k in n_pool})

print(f"X종 균형 대상: {len(x_needed)}개 / {sum(x_needed.values())}회")
print(f"N종 균형 대상: {len(n_needed)}개 / {sum(n_needed.values())}회")
print(f"총 필요 출현 : {sum(combined_needed.values())}회")
print(f"예상 이미지  : ~{sum(combined_needed.values())//3}장")

task2_coco = run_synthesis(
    combined_needed = combined_needed,
    all_pools       = all_pools,
    all_cyc         = all_cyc,
    bg_paths        = bg_paths,
    n_cats          = n_cats,
    cat2label       = combined_cat2label,
    x_size_stats    = None,
    out_img         = TASK2_IMG,
    out_ann         = TASK2_ANN,
    max_images      = 1200,
    seed            = 42,
)

pool: 56개 클래스, 총 2016개
pool: 18개 클래스, 총 1745개
X종 균형 대상: 53개 / 1323회
N종 균형 대상: 18개 / 630회
총 필요 출현 : 1953회
예상 이미지  : ~651장
[611/1200] 남은: 0개
[완료] 모든 클래스 목표 달성 (611장)

합성 완료: 611장
저장: /content/drive/MyDrive/(((((((내꺼)))))) 코드잇/스프린트 미션&피드백/초급프로젝트/dataset/task2_synthesized/annotations/_annotations.coco.json


## 셀 11. 결과 요약

In [27]:
label2name = {v:k for k,v in combined_cat2label.items()}
gen = Counter(label2name.get(a["category_id"],str(a["category_id"]))
              for a in task2_coco["annotations"])

print("="*55)
print("Task 2 합성 완료")
print("="*55)
print(f"생성 이미지: {len(task2_coco['images'])}장")
print(f"생성 박스  : {len(task2_coco['annotations'])}개")

print(f"\n{'class':>10} {'원본':>6} {'합성':>6} {'합계':>6} {'상태':>6}")
print("-"*40)
ok=warn=0
all_cats = sorted(set(list(x_current.keys())+list(n_needed.keys())))
for name in all_cats:
    orig=x_current.get(name,0); g=gen.get(name,0); total=orig+g
    flag="OK" if total>=TARGET_COUNT else "LOW"
    if total>=TARGET_COUNT: ok+=1
    else: warn+=1
    print(f"  {name:>8} {orig:>6} {g:>6} {total:>6}  {flag}")
print(f"\n정상({TARGET_COUNT}회+): {ok}개 / 미달: {warn}개")

Task 2 합성 완료
생성 이미지: 611장
생성 박스  : 1953개

     class     원본     합성     합계     상태
----------------------------------------
     10221      0     35     35  OK
     10224      0     35     35  OK
     12081      3     32     35  OK
     12247      3     32     35  OK
     12420      0     35     35  OK
     12778      6     29     35  OK
     13395      3     32     35  OK
     13900     15     20     35  OK
     16232     21     14     35  OK
     16262     23     12     35  OK
     16548     18     17     35  OK
     16551      3     32     35  OK
     16688      8     27     35  OK
     18110      0     35     35  OK
     18147     15     20     35  OK
     18357     13     22     35  OK
      1900     15     20     35  OK
     19232     17     18     35  OK
     19552      3     32     35  OK
     19607      6     29     35  OK
     19861      6     29     35  OK
     20014     17     18     35  OK
     20238     20     15     35  OK
     20877     12     23     35  OK
     21026    

In [ ]:
import json, os, glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from collections import defaultdict

with open(os.path.join(TASK2_ANN, "_annotations.coco.json"), encoding="utf-8") as f:
    t2_coco = json.load(f)

t2_id2img  = {img["id"]: img for img in t2_coco["images"]}
t2_id2name = {cat["id"]: cat["name"] for cat in t2_coco["categories"]}
t2_img2ann = defaultdict(list)
for ann in t2_coco["annotations"]:
    t2_img2ann[ann["image_id"]].append(ann)

N_COLS  = 5
CHUNK   = 50
img_ids = sorted(t2_id2img.keys())
total   = len(img_ids)
print(f"총 합성 이미지: {total}장")

for start in range(0, total, CHUNK):
    batch    = img_ids[start:start+CHUNK]
    n_rows   = (len(batch) + N_COLS - 1) // N_COLS
    fig, axes = plt.subplots(n_rows, N_COLS,
                              figsize=(N_COLS*3, n_rows*3))
    axes = axes.flatten()
    fig.suptitle(f"Task2 synthesized [{start+1}~{min(start+CHUNK,total)}/{total}]",
                 fontsize=12)

    for i, img_id in enumerate(batch):
        ax      = axes[i]
        img_info= t2_id2img[img_id]
        img_path= os.path.join(TASK2_IMG, img_info["file_name"])
        img     = Image.open(img_path).convert("RGB")
        ax.imshow(img)

        for ann in t2_img2ann[img_id]:
            x, y, w, h = ann["bbox"]
            cat_name    = t2_id2name.get(ann["category_id"], str(ann["category_id"]))
            rect = patches.Rectangle((x,y), w, h,
                                      linewidth=1.5, edgecolor="cyan", facecolor="none")
            ax.add_patch(rect)
            ax.text(x, y-3, cat_name, fontsize=5, color="cyan",
                    bbox=dict(facecolor="black", alpha=0.4, pad=1, edgecolor="none"))

        ax.set_title(img_info["file_name"], fontsize=5)
        ax.axis("off")

    for i in range(len(batch), len(axes)):
        axes[i].axis("off")

    plt.tight_layout()
    plt.show()
    print(f"[{start+1}~{min(start+CHUNK,total)}] 완료")